# House Price Linear Models

## Notebook 01: Data Understanding

### Objective

This notebook loads and validates the Melbourne Housing Snapshot dataset before exploratory analysis and modelling. It examines the dataset structure, column types, missing values, duplicates, and target validity without modifying the original raw data.

## 1. Import Required Libraries



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


## 2. Define and Validate the Dataset Path

The raw dataset should remain unchanged inside the `data/raw` directory. This step creates the dataset path and confirms that the file is available before attempting to load it.

In [2]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "melb_data.csv"


## 3. Load the Raw Dataset

The original CSV file is loaded into a pandas DataFrame named `df`. The dataset shape is displayed to confirm the number of rows and columns loaded.

In [3]:
df = pd.read_csv(data_path)

df.shape

(13580, 21)

### Observation

The raw dataset loaded successfully with 13,580 rows and 21 columns.

## 4. Preview the Dataset

The first five rows are displayed to inspect the column names, sample values, and overall structure of the raw dataset.

In [4]:
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


### Observation

The preview shows a mixture of numerical and categorical columns, with `Price` present as the target. `NaN` values are visible in fields such as `BuildingArea` and `YearBuilt`, indicating that missing data requires systematic investigation. The middle columns are truncated in the preview and must be listed separately.

## 5. Inspect Column Names

The complete list of column names is displayed to verify the available variables and identify the target column without relying on the truncated dataset preview.

In [5]:
df.columns.tolist()

['Suburb',
 'Address',
 'Rooms',
 'Type',
 'Price',
 'Method',
 'SellerG',
 'Date',
 'Distance',
 'Postcode',
 'Bedroom2',
 'Bathroom',
 'Car',
 'Landsize',
 'BuildingArea',
 'YearBuilt',
 'CouncilArea',
 'Lattitude',
 'Longtitude',
 'Regionname',
 'Propertycount']

### Observation

The dataset contains 21 named columns, and `Price` is confirmed as the target variable. The remaining columns describe property, sale, and location characteristics, but their stored data types require separate verification.

## 6. Inspect Data Types and Non-Null Counts

The dataset summary is examined to identify how each column is stored, compare non-null counts, and detect columns that may contain missing values.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

### Observation

The dataset contains 13 numerical columns and 8 object columns. The target `Price` has no missing values. Missing values are present in `Car` (62), `BuildingArea` (6,450), `YearBuilt` (5,375), and `CouncilArea` (1,369). The `Date` column is currently stored as an object rather than a datetime variable.

## 7. Quantify Missing Values

Missing counts and percentages are calculated for each column. Percentages provide context about the severity of missing data and will support later imputation decisions.

In [7]:
missing_count = df.isna().sum()
missing_percent = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percentage", ascending=False)

missing_summary

,Missing Count,Missing Percentage
BuildingArea,6450,47.50
YearBuilt,5375,39.58
CouncilArea,1369,10.08
Car,62,0.46


### Observation

Four columns contain missing values. `BuildingArea` has the highest missing percentage at 47.50%, followed by `YearBuilt` at 39.58%. `CouncilArea` has 10.08% missing values, while `Car` has only 0.46%. These values require further investigation before selecting an imputation or exclusion strategy.

## 8. Check for Exact Duplicate Rows

Exact duplicate rows can repeat the same information and may distort analysis or model evaluation. This step counts duplicates without removing them.

In [8]:
duplicate_count = df.duplicated().sum()
duplicate_percent = round(duplicate_count / len(df) * 100, 2)

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", duplicate_percent)

Duplicate rows: 0
Duplicate percentage: 0.0


### Observation

No exact duplicate rows were found in the dataset. Therefore, no row removal is required based on the exact duplicate check.

## 9. Validate the Target Variable

The target variable `Price` is checked for missing, non-finite, zero, and negative values. Its minimum and maximum are also reported to confirm the observed range before modelling.

In [9]:
target = "Price"

target_validation = pd.Series({
    "Missing values": df[target].isna().sum(),
    "Non-finite values": (~np.isfinite(df[target])).sum(),
    "Zero or negative values": (df[target] <= 0).sum(),
    "Minimum price": df[target].min(),
    "Maximum price": df[target].max()
})

target_validation

Missing values                   0.0
Non-finite values                0.0
Zero or negative values          0.0
Minimum price                85000.0
Maximum price              9000000.0
dtype: float64

### Observation

The `Price` target contains no missing, non-finite, zero, or negative values. Observed prices range from 85,000 to 9,000,000. The target is numerically valid for modelling, although its wide range requires later distribution and extreme-value investigation.

## 10. Inspect Column Cardinality

The number of distinct non-missing values in each column is calculated. This helps identify low-cardinality categories, high-cardinality variables, and possible identifier-like features before preprocessing.

In [10]:
unique_counts = df.nunique(dropna=True).sort_values(ascending=False)

unique_counts

Address          13378
Longtitude        7063
Lattitude         6503
Price             2204
Landsize          1448
BuildingArea       602
Suburb             314
Propertycount      311
SellerG            268
Distance           202
Postcode           198
YearBuilt          144
Date                58
CouncilArea         33
Bedroom2            12
Car                 11
Bathroom             9
Rooms                9
Regionname           8
Method               5
Type                 3
dtype: int64

### Observation

`Address` has the highest cardinality with 13,378 distinct values across 13,580 rows, making it a possible identifier-like feature. `Suburb` and `SellerG` are relatively high-cardinality categorical variables, while `Type`, `Method`, and `Regionname` have low cardinality. High cardinality in continuous numerical variables such as coordinates is expected and does not imply categorical-encoding problems.

## 11. Separate Features and Target

The target `Price` is separated from the input features before data splitting and preprocessing. This prevents the true target values from being included as model inputs.

In [11]:
X = df.drop(columns=target)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target present in X:", target in X.columns)

X shape: (13580, 20)
y shape: (13580,)
Target present in X: False


### Observation

`X` contains 13,580 rows and 20 predictor columns, while `y` contains 13,580 target values. `Price` is not present in `X`, confirming that feature-target separation was completed correctly.

## 12. Create a Fixed Train/Test Split

The data is split before any learned preprocessing is applied. Eighty percent is assigned to training and twenty percent to the final test set. A fixed random state makes the split reproducible.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (10864, 20)
X_test shape: (2716, 20)
y_train shape: (10864,)
y_test shape: (2716,)


### Observation

The fixed split contains 10,864 training rows and 2,716 test rows, which total 13,580 rows. Both training and test feature sets contain the same 20 predictor columns. The test set represents 20% of the dataset and will remain untouched until final evaluation.

## 13. Save Fixed Split Indices

The original row indices for the training and test sets are saved locally. Future notebooks can reload the raw dataset and use these index files to reproduce the exact same split.

In [13]:
processed_path = project_root / "data" / "processed"

pd.DataFrame({"row_index": X_train.index}).to_csv(
    processed_path / "train_indices.csv",
    index=False
)

pd.DataFrame({"row_index": X_test.index}).to_csv(
    processed_path / "test_indices.csv",
    index=False
)

print("Training indices saved:", len(X_train))
print("Test indices saved:", len(X_test))

Training indices saved: 10864
Test indices saved: 2716


## 14. Save Missing-Value Summary

The missing-value summary is saved as a CSV table so that the data-quality findings can be referenced outside the notebook.

In [15]:
tables_path = project_root / "results" / "tables"

missing_summary.to_csv(
    tables_path / "missing_value_summary.csv"
)

print("Saved:", tables_path / "missing_value_summary.csv")

Saved: C:\Users\mamona\Desktop\house-price-linear-models\results\tables\missing_value_summary.csv


## 15. Data Understanding Summary

### Key Findings

- The dataset contains 13,580 rows and 21 columns.
- `Price` is the target variable and contains no missing, non-finite, zero, or negative values.
- The dataset contains 13 numerical columns and 8 object columns.
- Exact duplicate rows were not found.
- Missing values are present in `Car`, `BuildingArea`, `YearBuilt`, and `CouncilArea`.
- `BuildingArea` has the highest missing percentage at 47.50%.
- `Address` has very high cardinality with 13,378 distinct values.
- The dataset was separated into 10,864 training rows and 2,716 test rows using a fixed random state.
- Training and test indices were saved locally for reproducibility.

### Next Step

The next notebook will perform exploratory data analysis and investigate distributions, relationships, missing-data patterns, categorical frequencies, outliers, and other data-quality characteristics using the training data.